In [1]:
from email import utils
import polars as pl
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import re
from tqdm import tqdm
sys.path.append("../src")  # Add path to where 'retail' package lives
import retail.retailfilter as rf # Correct import path
import retail.retailfilter_new as rfn
import utils.utils as ut  # Correct import path


def polars_to_geoparquet(
    pl_df: pl.DataFrame,
    out_path: str,
    lat_col: str = "lat",
    lng_col: str = "lng",
    crs: str = "EPSG:4326",
):
    pdf = pl_df.to_pandas()
    gdf = gpd.GeoDataFrame(
        pdf,
        geometry=gpd.points_from_xy(pdf[lng_col], pdf[lat_col]),
        crs=crs,
    )
    gdf.to_parquet(out_path, index=False)


[nltk_data] Downloading package names to /home/ogoodwin/nltk_data...
[nltk_data]   Package names is already up-to-date!


In [4]:
# input_file = Path("../data/input_files/uk_foursquare_datefields_admin_categories.parquet")
# input_file = Path("../data/input_files/uk_foursquare_cc_aug.parquet")
input_file = Path("../data/input_files/uk_foursquare_cc_nov_iceberg.parquet")
# Read in parquet
foursquare = pl.read_parquet(input_file)
print("original length:", len(foursquare))
# Filter
foursquare_filtnew = rfn.filter_foursquare(foursquare)

unique_levels = (
    foursquare_filtnew
    .group_by("level1_category_name", "level2_category_name", "level3_category_name")
    .len()
    .rename({"len": "count"})
    .sort("count", descending=True)
)

unique_levels.write_csv("../data/POI_files/foursquare_unique_levels.csv")
print(len(foursquare_filtnew))


# Columns to keep
COLUMNS_TO_KEEP = [
    "fsq_place_id", "name", "lat", "lng","fsq_category_labels", "address", "locality", "region",
    "postcode", "admin_region", "post_town", 
    "date_created", "date_refreshed", "date_closed", "tel", "website", 
    "fsq_category_ids", "h3_11","level1_category_name", "level2_category_name", "level3_category_name",
]

# name has changed now its exploded.
if "fsq_category_ids" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_id" else "fsq_category_ids" for col in COLUMNS_TO_KEEP]
elif "fsq_category_id" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_ids" else "fsq_category_id" for col in COLUMNS_TO_KEEP]
else:
    COLUMNS_TO_KEEP_MOD = COLUMNS_TO_KEEP

# Add H3 column
foursquare_filtnew = ut.add_h3_11_column(foursquare_filtnew)

# Keep only selected columns
foursquare_filtnew = foursquare_filtnew.select(COLUMNS_TO_KEEP_MOD)


#now do inversion method
print("Inversion method filtering")


# Read in parquet
foursquare = pl.read_parquet(input_file)
print("original length:", len(foursquare))
# Filter
foursquare_filtnew_inv = rfn.filter_foursquare_invmethod(foursquare)

unique_levels = (
    foursquare_filtnew_inv
    .group_by("level1_category_name", "level2_category_name", "level3_category_name")
    .len()
    .rename({"len": "count"})
    .sort("count", descending=True)
)

unique_levels.write_csv("../data/POI_files/foursquare_unique_levels_inv.csv")
print(len(foursquare_filtnew_inv))
#version sorted by name levels
unique_levels = unique_levels.sort(by=["level1_category_name", "level2_category_name", "level3_category_name"])
unique_levels.write_csv("../data/POI_files/foursquare_unique_levels_inv_sorted.csv")
# Columns to keep
COLUMNS_TO_KEEP = [
    "fsq_place_id", "name", "lat", "lng","fsq_category_labels", "address", "locality", "region",
    "postcode", "admin_region", "post_town", 
    "date_created", "date_refreshed", "date_closed", "tel", "website", 
    "fsq_category_ids", "h3_11","level1_category_name", "level2_category_name", "level3_category_name",
]

# name has changed now its exploded.
if "fsq_category_ids" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_id" else "fsq_category_ids" for col in COLUMNS_TO_KEEP]
elif "fsq_category_id" in foursquare_filtnew.columns:
    COLUMNS_TO_KEEP_MOD = [col if col != "fsq_category_ids" else "fsq_category_id" for col in COLUMNS_TO_KEEP]
else:
    COLUMNS_TO_KEEP_MOD = COLUMNS_TO_KEEP

# Add H3 column
foursquare_filtnew_inv = ut.add_h3_11_column(foursquare_filtnew_inv)

# Keep only selected columns
foursquare_filtnew_inv = foursquare_filtnew_inv.select(COLUMNS_TO_KEEP_MOD)




#get rows in filtered out by inversion method but not by new method
foursquare_excluded_invonly = foursquare_filtnew.join(
    foursquare_filtnew_inv,
    on="fsq_place_id",
    how="anti"  # Anti-join keeps rows that DON'T match
)
foursquare_excluded_invonly




original length: 4695350
Filtered Foursquare data (Retail Cats): 1938605 rows remaining.
Filtered Foursquare data (remove post box): 1938552 rows remaining.
Filtered Foursquare data (remove closed): 1666321 rows remaining.
1499296
Inversion method filtering
original length: 4695350
Filtered Foursquare data (Retail Cats): 1862176 rows remaining.
Filtered Foursquare data (remove post box): 1862123 rows remaining.
Filtered Foursquare data (remove closed): 1602161 rows remaining.
1437405


fsq_place_id,name,lat,lng,fsq_category_labels,address,locality,region,postcode,admin_region,post_town,date_created,date_refreshed,date_closed,tel,website,fsq_category_id,h3_11,level1_category_name,level2_category_name,level3_category_name
str,str,f64,f64,list[str],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""4e0ed07022711665f618b56f""","""Norton Way Mazda""",51.982068,-0.223569,"[""Retail > Automotive Retail > Car Dealership"", ""Business and Professional Services > Automotive Service > Automotive Repair Shop""]","""Norton Way N""","""Letchworth""","""Hertfordshire""","""SG6 1BL""","""England""",null,"""2011-07-02""","""2025-09-21""",null,"""0844 249 3067""","""http://www.mazda.co.uk""","""4eb1c1623b7b52c0e1adc2ec""","""8b194e495300fff""","""Retail""","""Automotive Retail""","""Car Dealership"""
"""4e9585990aaf41f39182b90a""","""Stoneacre Durham""",54.74983,-1.630954,"[""Retail > Automotive Retail > Car Dealership"", ""Business and Professional Services > Automotive Service > Automotive Repair Shop""]","""Sawmills Lane""","""Durham""","""DH""","""DH7 8AB""","""England""",null,"""2011-10-12""","""2021-04-11""",null,"""0191 378 0866""","""http://www.stoneacre.co.uk/dea…","""4eb1c1623b7b52c0e1adc2ec""","""8b1946584800fff""","""Retail""","""Automotive Retail""","""Car Dealership"""
"""52de25b4498ee13e6b20b819""","""Penfold Motors Ltd""",51.461853,-0.007412,"[""Retail > Automotive Retail > Car Dealership""]",null,null,null,null,null,null,"""2014-01-21""","""2014-01-25""",null,null,null,"""4eb1c1623b7b52c0e1adc2ec""","""8b194ad05bb6fff""","""Retail""","""Automotive Retail""","""Car Dealership"""
"""62e92f981290854caa60ca7d""","""BAM Specialist Cars""",53.167586,-1.392968,"[""Retail > Automotive Retail > Car Dealership""]","""Unit 4B Railway View""","""Chesterfield""","""Derbyshire""","""S45 9FR""",null,null,"""2022-08-02""","""2022-08-02""",null,"""01246 807075""",null,"""4eb1c1623b7b52c0e1adc2ec""","""8b194228cb99fff""","""Retail""","""Automotive Retail""","""Car Dealership"""
"""751cc84731224187bac35376""","""The Roxton Vehicle Centre""",52.175061,-0.340893,"[""Retail > Automotive Retail > Car Dealership > Used Car Dealership""]","""High Barns Farm Business Unit …","""Roxton""","""Bedfordshire""","""MK44 3ET""","""England""","""Bedford""","""2015-11-29""","""2025-06-12""",null,"""01234 870008""",null,"""5e8f501a03c7a9000c1e2e88""","""8b195d974d40fff""","""Retail""","""Automotive Retail""","""Car Dealership"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""8d5a0ec9f6fd4171aec6d694""","""Redbox Developments""",52.316413,-0.590544,"[""Business and Professional Services > Real Estate Service > Property Management Office"", ""Business and Professional Services > Real Estate Service > Real Estate Agency"", ""Business and Professional Services > Real Estate Service > Real Estate Development and Title Company""]","""10 Mallard Close""","""Higham Ferrers""","""Northamptonshire""","""NN10 8JQ""","""England""","""Rushden""","""2015-09-14""","""2025-08-01""",null,"""01933 650449""","""http://www.redboxdevelopments.…","""5032885091d4c4b30a586d66""","""8b194364132afff""","""Business and Professional Serv…","""Real Estate Service""","""Real Estate Agency"""
"""9c20fe002f1440fdec079cad""","""Racing Green Cars""",51.151988,-0.960964,"[""Retail > Automotive Retail > Car Dealership > Used Car Dealership""]","""Caker Stream Rd Unit 19""","""Alton""","""Hampshire""","""GU34 2QF""","""England""","""Alton""","""2019-01-08""","""2019-01-08""",null,"""01252 544888""","""http://www.racinggreencars.com""","""5e8f501a03c7a9000c1e2e88""","""8b1959951446fff""","""Retail""","""Automotive Retail""","""Car Dealership"""
"""6f40dbac4d5944d4afd0c189""","""Michael Burbridge""",51.518563,-0.728692,"[""Business and Professional Services > Advertising Agency"", ""Retail > Print Store""]","""Kings Grove Unit 23-25""","""Maidenhead""","""Berkshire""","""SL6 4DP""","""England""","""Maidenhead""","""2010-05-28""","""2010-05-28""",null,"""01628 773920"""

In [3]:
labels_counts = (
    foursquare_filtnew_inv
    .drop_nulls(subset=["fsq_category_labels"])
    .group_by("fsq_category_labels")
    .agg(pl.count().alias("count"))
    .sort("count", descending=True)
)
# First, convert the list column to a string representation for CSV
labels_counts_csv = labels_counts.with_columns(
    pl.col("fsq_category_labels").list.join(", ").alias("fsq_category_labels")
)

# Save the full version
labels_counts_csv.write_csv("../data/POI_files/foursquare_labels_counts.csv")

# Filter for rows with 2+ entries and save
labels_counts_multi = (
    labels_counts
    .filter(pl.col("fsq_category_labels").list.len() >= 2)
    .with_columns(
        pl.col("fsq_category_labels").list.join(", ").alias("fsq_category_labels")
    )
)

labels_counts_multi.write_csv("../data/POI_files/foursquare_labels_counts_multi.csv")

# Unpack individual labels and rank them
labels_unpacked = (
    foursquare_filtnew_inv
    .drop_nulls(subset=["fsq_category_labels"])
    .select("fsq_category_labels")
    .explode("fsq_category_labels")  # Unpack the lists
    .group_by("fsq_category_labels")
    .agg(pl.count().alias("count"))
    .sort("count", descending=True)
)

labels_unpacked.write_csv("../data/POI_files/foursquare_labels_unpacked_counts.csv")

print(f"Total unique label combinations: {len(labels_counts_csv)}")
print(f"Combinations with 2+ labels: {len(labels_counts_multi)}")
print(f"Total unique individual labels: {len(labels_unpacked)}")





Total unique label combinations: 26721
Combinations with 2+ labels: 26279
Total unique individual labels: 904


/tmp/ipykernel_36994/3424305630.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  .agg(pl.count().alias("count"))
/tmp/ipykernel_36994/3424305630.py:34: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
  .agg(pl.count().alias("count"))


In [ ]:
# get rows with "Business and Professional Services > Advertising Agency

foursquare_advagency = foursquare_filtnew.filter(
    pl.col("fsq_category_labels").list.contains("Business and Professional Services > Advertising Agency")
)
#save to geoparquet
polars_to_geoparquet(
    foursquare_advagency,
    "../data/POI_files/singlecattests/foursquare_adv_agency.parquet"
)



foursquare_printservice = foursquare_filtnew.filter(
    pl.col("fsq_category_labels").list.contains("Retail > Print Store")
)
foursquare_printservice
#save to geoparquet
polars_to_geoparquet(
    foursquare_printservice,
    "../data/POI_files/singlecattests/foursquare_print_service.parquet"
)


#Business and Professional Services > Photography Service > Photography Lab
foursquare_photolab = foursquare_filtnew.filter(
    pl.col("fsq_category_labels").list.contains("Business and Professional Services > Photography Service > Photography Lab")
)
foursquare_photolab
#save to geoparquet
polars_to_geoparquet(
    foursquare_photolab,
    "../data/POI_files/singlecattests/foursquare_photography_lab.parquet"
)

# Business and Professional Services > Real Estate Service > Real Estate Development and Title Company

foursquare_realestate = foursquare_filtnew.filter(
    pl.col("fsq_category_labels").list.contains("Business and Professional Services > Real Estate Service > Real Estate Development and Title Company")
)
foursquare_realestate
#save to geoparquet
polars_to_geoparquet(
    foursquare_realestate,
    "../data/POI_files/singlecattests/foursquare_real_estate_dev_title.parquet"
)

#Arts and Entertainment > Public Art,481
foursquare_publicart = foursquare_filtnew.filter(
    pl.col("fsq_category_labels").list.contains("Arts and Entertainment > Public Art")
)
foursquare_publicart
#save to geoparquet
polars_to_geoparquet(
    foursquare_publicart,
    "../data/POI_files/singlecattests/foursquare_public_art.parquet"
)

#waterparks

foursquare_waterparks = foursquare_filtnew.filter(
    pl.col("fsq_category_labels").list.contains("Arts and Entertainment > Water Park")
)

#save to geoparquet
polars_to_geoparquet(
    foursquare_waterparks,
    "../data/POI_files/singlecattests/foursquare_water_parks.parquet"
)

#Psychic and Astrologer
foursquare_psychic = foursquare_filtnew.filter(
    pl.col("fsq_category_labels").list.contains("Arts and Entertainment > Psychic and Astrologer")
)
polars_to_geoparquet(
    foursquare_psychic,
    "../data/POI_files/singlecattests/foursquare_psychic.parquet"
)

#Martial Arts Dojo
foursquare_martialarts = foursquare_filtnew.filter(
    pl.col("fsq_category_labels").list.contains("Sports and Recreation > Martial Arts Dojo")
)
#save to geoparquet
polars_to_geoparquet(
    foursquare_martialarts,
    "../data/POI_files/singlecattests/foursquare_martial_arts_dojo.parquet"
)

# Medical Supplies Store
foursquare_medicalsupp = foursquare_filtnew.filter(
    pl.col("fsq_category_labels").list.contains("Retail > Medical Supply Store")
)
#save to geoparquet
polars_to_geoparquet(
    foursquare_medicalsupp,
    "../data/POI_files/singlecattests/foursquare_medical_supply_store.parquet"
)

In [5]:
foursquare_printservice

fsq_place_id,name,lat,lng,fsq_category_labels,address,locality,region,postcode,admin_region,post_town,date_created,date_refreshed,date_closed,tel,website,fsq_category_id,h3_11,level1_category_name,level2_category_name,level3_category_name
str,str,f64,f64,list[str],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""58eb22e5247a4ef465fda55a""","""Ministry of Signs""",50.459612,-4.708997,"[""Retail > Print Store""]","""32e Normandy Way""","""Bodmin""","""Cornwall and Isles of Scilly""","""PL31 1EX""","""England""","""Bodmin""","""2015-10-15""","""2025-06-05""",null,"""01208 72170""",null,"""52f2ab2ebcbc57f1066b8b28""","""8b18740ea72cfff""","""Retail""","""Print Store""",null
"""6d176075c5df4c6f107cda07""","""Creative Signs""",50.217799,-5.280446,"[""Retail > Print Store""]","""Dolcoath Industrial Park, Dolc…","""Camborne""","""Cornwall and Isles of Scilly""","""TR14 8RA""","""England""","""Camborne""","""2012-12-27""","""2025-08-01""",null,"""01209 718681""","""http://www.creativesignsuk.com""","""52f2ab2ebcbc57f1066b8b28""","""8b187468b489fff""","""Retail""","""Print Store""",null
"""770e20786b7548a0289e358e""","""JMB P""",51.45869,-0.10309,"[""Retail > Print Store""]","""250 Milkwood Rd. Unit 25""","""London""","""Greater London""","""SE24 0HG""","""England""","""London""","""2017-02-05""","""2025-08-01""",null,"""020 7928 5765""",null,"""52f2ab2ebcbc57f1066b8b28""","""8b194ad11851fff""","""Retail""","""Print Store""",null
"""bae3302ff35746d62a99e483""","""J Connor Signs""",52.478415,-1.867876,"[""Retail > Print Store""]","""St Andrews Industrial Estate, …","""Birmingham""","""West Midlands""","""B9 4QB""","""England""","""Birmingham""","""2016-03-04""","""2025-06-15""",null,"""0121 773 1477""","""http://www.jconnorsigns.co.uk""","""52f2ab2ebcbc57f1066b8b28""","""8b195c059b08fff""","""Retail""","""Print Store""",null
"""79f4d5c6dab14f1715d57703""","""Daleba Evolution""",51.800648,-0.063979,"[""Retail > Print Store""]","""Tamworth Rd Unit 49""","""Hertford""","""Hertfordshire""","""SG13 7DJ""","""England""","""Hertford""","""2016-03-26""","""2016-03-26""",null,"""01536 525215""","""http://www.daleba.co.uk""","""52f2ab2ebcbc57f1066b8b28""","""8b194e4dd6d9fff""","""Retail""","""Print Store""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""599eafc2c0cacb47a086dba8""","""Butterfly Studios""",52.666963,1.284811,"[""Business and Professional Services > Photography Service > Photography Lab"", ""Retail > Print Store""]","""Hurricane Way""","""Norwich""","""Norfolk""","""NR6 6EZ""","""England""",null,"""2017-08-24""","""2025-03-24""",null,"""07914 630328""","""http://www.butterflystudios.uk…","""4eb1bdde3b7b55596b4a7490""","""8b1941053006fff""","""Business and Professional Serv…","""Photography Service""","""Photography Lab"""
"""0caf3ea6b18d4f209e6ed26c""","""John Brailsford Printers""",53.549587,-1.47561,"[""Retail > Print Store""]","""44 Doncaster Rd""","""Barnsley""","""South Yorkshire""","""S70 1TL""","""England""","""Barnsley""","""2010-05-22""","""2010-05-22""",null,"""01226 779999""","""http://www.brailsfordprinters.…","""52f2ab2ebcbc57f1066b8b28""","""8b194201d4dcfff""","""Retail""","""Print Store""",null
"""d30542e7c3754a5ee09725c0""","""Reaction Asset Management""",50.731088,-1.910989,"[""Business and Professional Services > Advertising Agency"", ""Retail > Print Store""]","""Dalling Rd Unit 1""","""Pool""","""Dorset""","""BH12 1DJ""","""England""","""Poole""","""2013-07-01""","""2013-07-01""",null,"""01202 757579""","""http://www.reaction-group.co.u…","""52f2ab2ebcbc57f1066b8b28""","""8b1959716759fff""","""Retail""","""Print Store""",null


In [4]:
malls = foursquare_filtnew.filter(
    pl.col("fsq_category_id").is_in(["5744ccdfe4b0c0459246b4dc", "4bf58dd8d48988d1fd941735"])
)

print(len(malls))
malls_unique = malls.unique(subset=["fsq_place_id"])


3043


In [3]:
#Medical Supply Store

#get rows which are level2 Medical Supply Store and 
warehouse = foursquare_filtnew.filter(pl.col("level2_category_name") == "Warehouse or Wholesale Store")
polars_to_geoparquet(warehouse ,"warehouse_or_wholesale_store.parquet")
#remove all "Warehouse or Wholesale Store" unless "Argos"
foursquare_filtnew = foursquare_filtnew.filter(
    (pl.col("level2_category_name") != "Warehouse or Wholesale Store") | (pl.col("name") == "Argos")
)



In [11]:
#Medical Supply Store

#get rows which are level2 Medical Supply Store and 
medical_supply = foursquare_filtnew.filter(pl.col("level2_category_name") == "Medical Supply Store")